In [1]:
import os
import shutil
import numpy
import xml.etree.ElementTree as ET
from pathlib import Path
from tqdm import tqdm

In [ ]:
VOC_CLASSES = [
    "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow",
    "diningtable", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "sofa", "train", "tvmonitor"
]
CLASS2ID = {c: i for i, c in enumerate(VOC_CLASSES)}

def convert_box(size_w, size_h, box):
    # box: (xmin, ymin, xmax, ymax) in VOC (1-based sometimes)
    xmin, ymin, xmax, ymax = box
    # YOLO: (x_center, y_center, w, h) normalized
    x = (xmin + xmax) / 2.0 / size_w
    y = (ymin + ymax) / 2.0 / size_h
    w = (xmax - xmin) / size_w
    h = (ymax - ymin) / size_h
    return x, y, w, h

def parse_voc_xml(xml_path: Path):
    root = ET.parse(xml_path).getroot()
    size = root.find("size")
    w = int(size.find("width").text)
    h = int(size.find("height").text)

    labels = []
    for obj in root.findall("object"):
        name = obj.find("name").text.strip()
        difficult = obj.find("difficult")
        difficult = int(difficult.text) if difficult is not None else 0

        if name not in CLASS2ID:
            continue

        bnd = obj.find("bndbox")
        xmin = float(bnd.find("xmin").text)
        ymin = float(bnd.find("ymin").text)
        xmax = float(bnd.find("xmax").text)
        ymax = float(bnd.find("ymax").text)

        xmin = max(0.0, xmin)
        ymin = max(0.0, ymin)
        xmax = max(0.0, xmax)
        ymax = max(0.0, ymax)

        x, y, bw, bh = convert_box(w, h, (xmin, ymin, xmax, ymax))
        cls_id = CLASS2ID[name]
        labels.append((cls_id, x, y, bw, bh, difficult))
    return labels

def make_yolo_dataset(voc_root, out_root):
    voc_root = Path(voc_root)
    out_root = Path(out_root)

    img_dir = voc_root / "JPEGImages"
    ann_dir = voc_root / "Annotations"
    set_dir = voc_root / "ImageSets" / "Main"

    for split in ["train", "val"]:
        (out_root / "images" / split).mkdir(parents=True, exist_ok=True)
        (out_root / "labels" / split).mkdir(parents=True, exist_ok=True)

    split_files = {
        "train": set_dir / "train.txt",
        "val": set_dir / "val.txt",
    }

    for split, list_path in split_files.items():
        if not list_path.exists():
            raise FileNotFoundError(f"Does not find {list_path}")

        ids = [x.strip() for x in list_path.read_text().splitlines() if x.strip()]
        print(f"{split}: {len(ids)} images")

        for img_id in tqdm(ids, desc=f"Converting {split}"):
            src_img = img_dir / f"{img_id}.jpg"
            src_xml = ann_dir / f"{img_id}.xml"

            if not src_img.exists() or not src_xml.exists():
                continue

            labels = parse_voc_xml(src_xml)

            label_txt = out_root / "labels" / split / f"{img_id}.txt"
            with open(label_txt, "w", encoding="utf-8") as f:
                for cls_id, x, y, bw, bh, difficult in labels:
                    f.write(f"{cls_id} {x:.6f} {y:.6f} {bw:.6f} {bh:.6f}\n")

            dst_img = out_root / "images" / split / f"{img_id}.jpg"
            if not dst_img.exists():
                shutil.copy2(src_img, dst_img)

if __name__ == "__main__":
    VOC_ROOT = r"C:\Users\Ru199\Desktop\ML project\OD\VOCdevkit2007\VOC2007"
    OUT_ROOT = r"C:\Users\Ru199\Desktop\ML project\OD\voc2007_yolo"

    make_yolo_dataset(VOC_ROOT, OUT_ROOT)
    print("Done. YOLO dataset at:", OUT_ROOT)